# 🔄 Data Preprocessing Pipeline
## Cross-Platform User Matching

Pipeline นี้แบ่งการทำงานออกเป็น **6 Stage** ที่แยกจากกันอย่างชัดเจน:

| Stage | ชื่อ | หน้าที่ |
|-------|------|--------|
| 1 | **Configuration & Imports** | ตั้งค่า path, import libraries |
| 2 | **Data Loading** | โหลด JSON profiles จาก Dataset-LinkSocial |
| 3 | **Data Cleaning & Standardization** | NaN handling, normalize text, remove emojis |
| 4 | **Platform Splitting** | แยก DataFrame ตาม platform |
| 5 | **Ground Truth & Training Data** | สร้าง positive/negative pairs สำหรับ training |
| 6 | **Export & Quality Report** | บันทึกไฟล์ + สรุปคุณภาพข้อมูล |

---

## Stage 1: Configuration & Imports
ตั้งค่า path, import libraries ที่จำเป็น

In [1]:
import os
import json
import re
import pandas as pd
import numpy as np
from typing import Dict, Tuple
from datetime import datetime

# ============================================================
# CONFIG — แก้ path ตรงนี้ที่เดียว
# ============================================================
BASE_PATH   = "../data/data/Dataset-LinkSocial"  # โฟลเดอร์ dataset ต้นทาง
OUTPUT_DIR  = "../data/processed"                 # โฟลเดอร์ output

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config loaded")
print(f"   BASE_PATH  = {os.path.abspath(BASE_PATH)}")
print(f"   OUTPUT_DIR = {os.path.abspath(OUTPUT_DIR)}")

✅ Config loaded
   BASE_PATH  = d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\data\Dataset-LinkSocial
   OUTPUT_DIR = d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\processed


---
## Stage 2: Data Loading
โหลด JSON profiles จากทั้ง 3 โฟลเดอร์ (`1.profile.data`, `2.profile.data`, `3.profile.data`)  
แต่ละไฟล์จะถูกระบุ platform จากชื่อไฟล์ (twitter / instagram / googleplus)

Step 2.1: **Load JSON Files** — โหลดไฟล์ JSON ทั้งหมด

In [ ]:
def load_all_profiles(base_path: str) -> pd.DataFrame:
    """
    โหลด profiles ทั้งหมดจาก 3 โฟลเดอร์ profile.data
    
    Returns:
        pd.DataFrame ที่มี column: userName, fullName, bio, location,
        externalUrl, pictureURL, platform, source_folder, user_folder
    """
    all_profiles = []
    profile_folders = ["1.profile.data", "2.profile.data", "3.profile.data"]
    
    for folder in profile_folders:
        folder_path = os.path.join(base_path, folder)
        
        if not os.path.exists(folder_path):
            print(f"⚠️  Folder {folder_path} not found, skipping...")
            continue
            
        print(f"📂 Loading from {folder}...")
        
        for user_folder in os.listdir(folder_path):
            user_path = os.path.join(folder_path, user_folder)
            
            if os.path.isdir(user_path):
                for file in os.listdir(user_path):
                    if file.endswith('.json'):
                        file_path = os.path.join(user_path, file)
                        try:
                            with open(file_path, 'r', encoding='utf-8') as f:
                                data = json.load(f)
                                data['source_folder'] = folder
                                data['user_folder'] = user_folder
                                
                                # ระบุ platform จากชื่อไฟล์
                                if 'twitter' in file.lower():
                                    data['platform'] = 'twitter'
                                elif 'instagram' in file.lower():
                                    data['platform'] = 'instagram'
                                elif 'google' in file.lower():
                                    data['platform'] = 'googleplus'
                                else:
                                    data['platform'] = 'unknown'
                                    
                                all_profiles.append(data)
                        except Exception as e:
                            print(f"❌ Error reading {file_path}: {e}")
    
    df = pd.DataFrame(all_profiles)
    
    print(f"\n{'='*50}")
    print(f"📦 Step 2.1 Summary")
    print(f"{'='*50}")
    print(f"Total JSON files loaded: {len(df)}")
    
    return df

# === โหลดไฟล์ทั้งหมด ===
df_raw = load_all_profiles(BASE_PATH)


📂 Loading from 1.profile.data...
📂 Loading from 2.profile.data...
📂 Loading from 3.profile.data...


Step 2.2: **Create DataFrame & Display** — สร้าง DataFrame และแสดงผล

In [ ]:
# === แสดงผล DataFrame ===
print(f"{'='*50}")
print(f"📊 Step 2.2: DataFrame Summary")
print(f"{'='*50}")
print(f"Total profiles : {len(df_raw)}")
print(f"Total columns  : {len(df_raw.columns)}")
print(f"Columns        : {list(df_raw.columns)}")

print(f"\n📈 Platform distribution:")
print(df_raw['platform'].value_counts().to_string())

print(f"\n📂 Source folder distribution:")
print(df_raw['source_folder'].value_counts().to_string())

print(f"\n📋 DataFrame Info:")
df_raw.info()

print(f"\n🔍 Sample data (first 5 rows):")
df_raw.head()


📂 Loading from 1.profile.data...
📂 Loading from 2.profile.data...
📂 Loading from 3.profile.data...

📊 Stage 2 Summary
Total profiles loaded: 24729
Columns: ['userName', 'fullName', 'bigrams', 'source_folder', 'user_folder', 'platform', 'bio', 'externalUrl', 'outputProfileName', 'pictureURL', 'location']

Platform distribution:
platform
twitter       8535
googleplus    8302
instagram     7892

🔍 Sample data (first 3 rows):


,userName,fullName,platform,user_folder
0,i3mawi,Adeeb Amawi,googleplus,A3mawi
1,WoltersKluwerEspaa,Wolters Kluwer Espaa,googleplus,A3Software
2,AALISHANMATRIX,AALISHAN MATRIX,googleplus,aalishanmatrix


Step 2.3 : **Save Raw CSV** — บันทึกเป็น CSV

In [ ]:
# === แสดงผล DataFrame ===
print(f"{'='*50}")
print(f"📊 Step 2.2: DataFrame Summary")
print(f"{'='*50}")
print(f"Total profiles : {len(df_raw)}")
print(f"Total columns  : {len(df_raw.columns)}")
print(f"Columns        : {list(df_raw.columns)}")

print(f"\n📈 Platform distribution:")
print(df_raw['platform'].value_counts().to_string())

print(f"\n📂 Source folder distribution:")
print(df_raw['source_folder'].value_counts().to_string())

print(f"\n📋 DataFrame Info:")
df_raw.info()

print(f"\n🔍 Sample data (first 5 rows):")
df_raw.head()


Step 2.4 : **Verify Saved CSV** — ตรวจสอบ CSV ที่บันทึก

In [ ]:
# === ตรวจสอบ CSV ที่บันทึก ===
df_verify = pd.read_csv(raw_csv_path)

print(f"{'='*50}")
print(f"🔍 Step 2.4: Verify Saved CSV")
print(f"{'='*50}")

# เปรียบเทียบจำนวน row/column
rows_match = len(df_verify) == len(df_raw)
cols_match = len(df_verify.columns) == len(df_raw.columns)

print(f"Rows   — Original: {len(df_raw):>6} | CSV: {len(df_verify):>6} | {'✅ Match' if rows_match else '❌ Mismatch'}")
print(f"Cols   — Original: {len(df_raw.columns):>6} | CSV: {len(df_verify.columns):>6} | {'✅ Match' if cols_match else '❌ Mismatch'}")

print(f"\n📈 Platform distribution (จาก CSV):")
print(df_verify['platform'].value_counts().to_string())

print(f"\n🔍 Sample data จาก CSV (first 5 rows):")
df_verify.head()


---
## Stage 3: Data Cleaning

### Step 3.1: Helper Functions
กำหนดฟังก์ชันพื้นฐานที่ใช้ในขั้นตอนถัดไป พร้อมทดสอบกับข้อมูลจริงจาก `df_raw`

In [ ]:
def remove_emojis(text: str) -> str:
    """ลบ emojis และ special unicode characters ออกจาก text"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "]+", 
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)


def normalize_text(text: str) -> str:
    """Normalize: ลบ emoji -> lowercase -> ลบ @ นำหน้า -> ลบ special chars -> trim"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = remove_emojis(text)
    text = text.lower()
    text = re.sub(r'^@', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


def clean_bio(text: str) -> str:
    """Clean bio: ลบ emoji -> ลบ URL -> normalize whitespace -> trim"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = remove_emojis(text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


def normalize_location(text: str) -> str:
    """Normalize location: ลบ emoji -> lowercase -> ลบ special chars (เก็บ comma) -> trim"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = remove_emojis(text)
    text = text.lower()
    text = re.sub(r'[^\w\s,]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


def normalize_url(text: str) -> str:
    """Normalize URL: trim -> lowercase -> ลบ protocol -> ลบ www. -> ลบ trailing /"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r'^https?://', '', text)
    text = re.sub(r'^www\.', '', text)
    text = text.rstrip('/')
    return text


# === ทดสอบกับข้อมูลจริงจาก df_raw ===
print("🔍 ตัวอย่าง remove_emojis() กับข้อมูลจริง:")
print("=" * 80)

demo_cols = ['userName', 'fullName', 'bio']
emoji_rows = []

for col in demo_cols:
    if col not in df_raw.columns:
        continue
    found = 0
    for val in df_raw[col].dropna().values:
        original = str(val)
        cleaned = remove_emojis(original)
        if original != cleaned and len(original.strip()) > 0:
            emoji_rows.append({'Column': col, 'Before': original[:60], 'After': cleaned[:60]})
            found += 1
            if found >= 3:
                break

if emoji_rows:
    print(pd.DataFrame(emoji_rows).to_string(index=False))
else:
    print("  ℹ️ ไม่พบ emoji ในตัวอย่าง")

print(f"\n🔍 ตัวอย่าง normalize_text() กับข้อมูลจริง:")
print("=" * 80)

norm_rows = []
for col in demo_cols:
    if col not in df_raw.columns:
        continue
    found = 0
    for val in df_raw[col].dropna().values:
        original = str(val)
        cleaned = normalize_text(original)
        if original != cleaned and len(original.strip()) > 0:
            norm_rows.append({'Column': col, 'Before': original[:60], 'After': cleaned[:60]})
            found += 1
            if found >= 3:
                break

if norm_rows:
    print(pd.DataFrame(norm_rows).to_string(index=False))

print(f"\n✅ Helper functions defined — พร้อมใช้งาน (6 functions)")

### Step 3.2: NaN Handling
จัดการค่า **NaN / missing values** ใน text fields ทั้งหมด  
- Text columns (`userName`, `fullName`, `bio`, `location`, `externalUrl`, `pictureURL`) → แทนด้วย `''`  
- `outputProfileName` → ถ้าเป็น NaN ใช้ `user_folder` แทน

In [20]:
# สร้าง copy เพื่อไม่แก้ df_raw
df_clean = df_raw.copy()

# --- ก่อนทำ: ดู NaN ทั้งหมด ---
text_columns = ['userName', 'fullName', 'bio', 'location', 'externalUrl', 'pictureURL']

print("📊 NaN Count — ก่อนทำ (Before):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        nan_count = df_clean[col].isna().sum()
        total = len(df_clean)
        pct = nan_count / total * 100
        print(f"  {col:15s}: {nan_count:>5} NaN ({pct:5.1f}%)")
    else:
        print(f"  {col:15s}: ❌ column ไม่มีใน DataFrame")

📊 NaN Count — ก่อนทำ (Before):
----------------------------------------
  userName       :     0 NaN (  0.0%)
  fullName       :   276 NaN (  1.1%)
  bio            :  3905 NaN ( 15.8%)
  location       : 16194 NaN ( 65.5%)
  externalUrl    :  3897 NaN ( 15.8%)
  pictureURL     :  8425 NaN ( 34.1%)


In [21]:
# --- ทำ NaN Handling ---

# Text fields: NaN → empty string
for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('')

# outputProfileName: ถ้าไม่มีให้ใช้ user_folder แทน
if 'outputProfileName' in df_clean.columns:
    df_clean['outputProfileName'] = df_clean['outputProfileName'].fillna(df_clean['user_folder'])
else:
    df_clean['outputProfileName'] = df_clean['user_folder']

# --- หลังทำ: ดู NaN อีกที ---
print("📊 NaN Count — หลังทำ (After):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        nan_count = df_clean[col].isna().sum()
        print(f"  {col:15s}: {nan_count:>5} NaN")

# ดูจำนวน empty string ด้วย
print(f"\n📊 Empty String Count (หลัง fillna):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        empty_count = (df_clean[col] == '').sum()
        total = len(df_clean)
        pct = empty_count / total * 100
        print(f"  {col:15s}: {empty_count:>5} empty ({pct:5.1f}%)")

print(f"\n✅ Step 3.2 NaN Handling เสร็จ")

📊 NaN Count — หลังทำ (After):
----------------------------------------
  userName       :     0 NaN
  fullName       :     0 NaN
  bio            :     0 NaN
  location       :     0 NaN
  externalUrl    :     0 NaN
  pictureURL     :     0 NaN

📊 Empty String Count (หลัง fillna):
----------------------------------------
  userName       :    61 empty (  0.2%)
  fullName       :   309 empty (  1.2%)
  bio            :  4379 empty ( 17.7%)
  location       : 16790 empty ( 67.9%)
  externalUrl    :  3897 empty ( 15.8%)
  pictureURL     :  8425 empty ( 34.1%)

✅ Step 3.2 NaN Handling เสร็จ


### Step 3.3: Normalize Username
`userName` → `userName_clean`

การแปลง:
- ลบ emoji → lowercase → ลบ `@` นำหน้า → ลบ special chars → trim  
- ตัวอย่าง: `@User.Name_01` → `username_01`

In [22]:
# --- Normalize userName ---
df_clean['userName_clean'] = df_clean['userName'].apply(normalize_text)

# === แสดงผลลัพธ์ ===
print("📊 Step 3.3: Normalize Username — ผลลัพธ์")
print("=" * 60)

# สถิติ
total = len(df_clean)
non_empty = (df_clean['userName_clean'].str.len() > 0).sum()
changed = (df_clean['userName'] != df_clean['userName_clean']).sum()
print(f"  Total profiles     : {total}")
print(f"  Non-empty username : {non_empty} ({non_empty/total*100:.1f}%)")
print(f"  Changed by normalization : {changed} ({changed/total*100:.1f}%)")

# ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน)
diff_mask = df_clean['userName'] != df_clean['userName_clean']
sample_diff = df_clean[diff_mask][['userName', 'userName_clean', 'platform']].head(10)

if len(sample_diff) > 0:
    print(f"\n🔍 ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน, สูงสุด 10 rows):")
    print("-" * 60)
    for _, row in sample_diff.iterrows():
        print(f"  [{row['platform']:10s}] '{row['userName']}' → '{row['userName_clean']}'")
else:
    print(f"\n  ℹ️ ไม่มี username ที่เปลี่ยนแปลง")

print(f"\n✅ Step 3.3 เสร็จ — เพิ่ม column: userName_clean")

📊 Step 3.3: Normalize Username — ผลลัพธ์
  Total profiles     : 24729
  Non-empty username : 24668 (99.8%)
  Changed by normalization : 16372 (66.2%)

🔍 ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน, สูงสุด 10 rows):
------------------------------------------------------------
  [googleplus] 'WoltersKluwerEspaa' → 'wolterskluwerespaa'
  [googleplus] 'AALISHANMATRIX' → 'aalishanmatrix'
  [twitter   ] '@aaronbird' → 'aaronbird'
  [googleplus] 'GuantanamoBae' → 'guantanamobae'
  [twitter   ] '@aaronzlewis' → 'aaronzlewis'
  [googleplus] 'AlexBindaFernndez' → 'alexbindafernndez'
  [googleplus] 'AbelSerral' → 'abelserral'
  [googleplus] 'AgustinCervantes' → 'agustincervantes'
  [twitter   ] '@AckleySuicide' → 'ackleysuicide'
  [twitter   ] '@adambonham' → 'adambonham'

✅ Step 3.3 เสร็จ — เพิ่ม column: userName_clean


### Step 3.4: Normalize FullName
`fullName` → `fullName_clean`

การแปลง:
- ลบ emoji → lowercase → ลบ special chars → trim  
- ตัวอย่าง: `John   Doe!!!` → `john doe`

In [23]:
# --- Normalize fullName ---
df_clean['fullName_clean'] = df_clean['fullName'].apply(normalize_text)

# === แสดงผลลัพธ์ ===
print("📊 Step 3.4: Normalize FullName — ผลลัพธ์")
print("=" * 60)

# สถิติ
total = len(df_clean)
non_empty = (df_clean['fullName_clean'].str.len() > 0).sum()
changed = (df_clean['fullName'] != df_clean['fullName_clean']).sum()
print(f"  Total profiles     : {total}")
print(f"  Non-empty fullName : {non_empty} ({non_empty/total*100:.1f}%)")
print(f"  Changed by normalization : {changed} ({changed/total*100:.1f}%)")

# ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน)
diff_mask = df_clean['fullName'] != df_clean['fullName_clean']
sample_diff = df_clean[diff_mask][['fullName', 'fullName_clean', 'platform']].head(10)

if len(sample_diff) > 0:
    print(f"\n🔍 ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน, สูงสุด 10 rows):")
    print("-" * 60)
    for _, row in sample_diff.iterrows():
        orig = str(row['fullName'])[:35]
        clean = str(row['fullName_clean'])[:35]
        print(f"  [{row['platform']:10s}] '{orig}' → '{clean}'")
else:
    print(f"\n  ℹ️ ไม่มี fullName ที่เปลี่ยนแปลง")

print(f"\n✅ Step 3.4 เสร็จ — เพิ่ม column: fullName_clean")

📊 Step 3.4: Normalize FullName — ผลลัพธ์
  Total profiles     : 24729
  Non-empty fullName : 24283 (98.2%)
  Changed by normalization : 23181 (93.7%)

🔍 ตัวอย่าง Before → After (แสดงเฉพาะที่เปลี่ยน, สูงสุด 10 rows):
------------------------------------------------------------
  [googleplus] 'Adeeb Amawi' → 'adeeb amawi'
  [googleplus] 'Wolters Kluwer Espaa' → 'wolters kluwer espaa'
  [googleplus] 'AALISHAN MATRIX' → 'aalishan matrix'
  [twitter   ] 'Aaron Bird' → 'aaron bird'
  [googleplus] 'Guantanamo Bae' → 'guantanamo bae'
  [twitter   ] 'aaron z. lewis' → 'aaron z lewis'
  [instagram ] 'O'Neil' → 'oneil'
  [googleplus] 'Alex Binda Fernndez' → 'alex binda fernndez'
  [googleplus] 'Abel Serral' → 'abel serral'
  [googleplus] 'Agustin Cervantes' → 'agustin cervantes'

✅ Step 3.4 เสร็จ — เพิ่ม column: fullName_clean


### Step 3.5: Clean Bio
`bio` → `bio_clean`

การแปลง:
- ลบ emoji และ special unicode  
- ลบ URL (http/https/www)  
- Normalize whitespace (รวม spaces/newlines ซ้ำ)  
- Trim  
- ตัวอย่าง: `Developer 🚀 https://dev.to | Bangkok` → `Developer  | Bangkok`

In [ ]:
# --- Clean Bio (enhanced) ---
df_clean['bio_clean'] = df_clean['bio'].apply(clean_bio)

print("📊 Step 3.5: Clean Bio — ผลลัพธ์")
print("=" * 60)

total = len(df_clean)
non_empty_orig = (df_clean['bio'].str.len() > 0).sum()
non_empty_clean = (df_clean['bio_clean'].str.len() > 0).sum()
changed = (df_clean['bio'] != df_clean['bio_clean']).sum()
print(f"  Total profiles       : {total}")
print(f"  Non-empty bio (orig) : {non_empty_orig} ({non_empty_orig/total*100:.1f}%)")
print(f"  Non-empty bio (clean): {non_empty_clean} ({non_empty_clean/total*100:.1f}%)")
print(f"  Changed              : {changed} ({changed/total*100:.1f}%)")

diff_mask = (df_clean['bio'] != df_clean['bio_clean']) & (df_clean['bio'].str.len() > 0)
sample_diff = df_clean[diff_mask][['bio', 'bio_clean', 'platform']].head(8)

if len(sample_diff) > 0:
    print(f"\n🔍 ตัวอย่าง Before → After (สูงสุด 8 rows):")
    print("-" * 60)
    for _, row in sample_diff.iterrows():
        orig = str(row['bio'])[:50]
        clean = str(row['bio_clean'])[:50]
        print(f"  [{row['platform']}]")
        print(f"    Before: '{orig}'")
        print(f"    After : '{clean}'")
        print()
else:
    print(f"\n  ℹ️ ไม่มี bio ที่เปลี่ยนแปลง")

bio_lengths = df_clean['bio_clean'].str.len()
print(f"📏 สถิติความยาว bio_clean:")
print(f"  Mean   : {bio_lengths.mean():.1f} chars")
print(f"  Median : {bio_lengths.median():.1f} chars")
print(f"  Max    : {bio_lengths.max()} chars")
print(f"  Empty  : {(bio_lengths == 0).sum()} profiles")

print(f"\n✅ Step 3.5 เสร็จ — เพิ่ม column: bio_clean")

### Step 3.6: Normalize Location
`location` → `location_clean`

การแปลง:
- ลบ emoji  
- Lowercase  
- ลบ special chars (เก็บ comma, space, underscore)  
- Normalize whitespace → trim  
- ตัวอย่าง: `New York, NY 🗽!!` → `new york, ny`

In [ ]:
# --- Normalize Location ---
df_clean['location_clean'] = df_clean['location'].apply(normalize_location)

print("📊 Step 3.6: Normalize Location — ผลลัพธ์")
print("=" * 60)

total = len(df_clean)
non_empty_orig = (df_clean['location'].str.len() > 0).sum()
non_empty_clean = (df_clean['location_clean'].str.len() > 0).sum()
changed = (df_clean['location'] != df_clean['location_clean']).sum()
print(f"  Total profiles          : {total}")
print(f"  Non-empty location (raw): {non_empty_orig} ({non_empty_orig/total*100:.1f}%)")
print(f"  Non-empty (clean)       : {non_empty_clean} ({non_empty_clean/total*100:.1f}%)")
print(f"  Changed by normalization: {changed} ({changed/total*100:.1f}%)")

diff_mask = (df_clean['location'] != df_clean['location_clean']) & (df_clean['location'].str.len() > 0)
sample_diff = df_clean[diff_mask][['location', 'location_clean', 'platform']].head(10)

if len(sample_diff) > 0:
    print(f"\n🔍 ตัวอย่าง Before → After (สูงสุด 10 rows):")
    print("-" * 60)
    for _, row in sample_diff.iterrows():
        print(f"  [{row['platform']:10s}] '{row['location'][:40]}' → '{row['location_clean'][:40]}'")
else:
    print(f"\n  ℹ️ ไม่มี location ที่เปลี่ยนแปลง")

print(f"\n✅ Step 3.6 เสร็จ — เพิ่ม column: location_clean")

### Step 3.7: Normalize External URL
`externalUrl` → `externalUrl_clean`

การแปลง:
- Trim → Lowercase  
- ลบ protocol (`http://`, `https://`)  
- ลบ `www.` นำหน้า  
- ลบ trailing `/`  
- ตัวอย่าง: `HTTPS://www.Example.COM/page/` → `example.com/page`

In [ ]:
# --- Normalize External URL ---
df_clean['externalUrl_clean'] = df_clean['externalUrl'].apply(normalize_url)

print("📊 Step 3.7: Normalize External URL — ผลลัพธ์")
print("=" * 60)

total = len(df_clean)
non_empty_orig = (df_clean['externalUrl'].str.len() > 0).sum()
non_empty_clean = (df_clean['externalUrl_clean'].str.len() > 0).sum()
changed = (df_clean['externalUrl'] != df_clean['externalUrl_clean']).sum()
print(f"  Total profiles              : {total}")
print(f"  Non-empty externalUrl (raw) : {non_empty_orig} ({non_empty_orig/total*100:.1f}%)")
print(f"  Non-empty (clean)           : {non_empty_clean} ({non_empty_clean/total*100:.1f}%)")
print(f"  Changed by normalization    : {changed} ({changed/total*100:.1f}%)")

diff_mask = (df_clean['externalUrl'] != df_clean['externalUrl_clean']) & (df_clean['externalUrl'].str.len() > 0)
sample_diff = df_clean[diff_mask][['externalUrl', 'externalUrl_clean', 'platform']].head(10)

if len(sample_diff) > 0:
    print(f"\n🔍 ตัวอย่าง Before → After (สูงสุด 10 rows):")
    print("-" * 60)
    for _, row in sample_diff.iterrows():
        print(f"  [{row['platform']:10s}] '{row['externalUrl'][:40]}' → '{row['externalUrl_clean'][:40]}'")
else:
    print(f"\n  ℹ️ ไม่มี URL ที่เปลี่ยนแปลง")

print(f"\n✅ Step 3.7 เสร็จ — เพิ่ม column: externalUrl_clean")

### Step 3.8: Create Profile ID
`outputProfileName` → `profile_id`

การแปลง:
- ใช้ `outputProfileName` (ใช้ `user_folder` เป็น fallback) แล้ว normalize  
- Profile ID นี้ใช้เป็น **ground truth** สำหรับจับคู่คนเดียวกันข้ามแพลตฟอร์ม

In [ ]:
# --- Create Profile ID ---
df_clean['profile_id'] = df_clean['outputProfileName'].apply(normalize_text)

# === แสดงผลลัพธ์ ===
print("📊 Step 3.8: Create Profile ID — ผลลัพธ์")
print("=" * 60)

# สถิติ
total = len(df_clean)
unique_ids = df_clean['profile_id'].nunique()
non_empty = (df_clean['profile_id'].str.len() > 0).sum()
print(f"  Total profiles     : {total}")
print(f"  Unique profile IDs : {unique_ids}")
print(f"  Non-empty IDs      : {non_empty} ({non_empty/total*100:.1f}%)")

# ดูว่ามี profile_id ที่ซ้ำข้ามแพลตฟอร์มกี่คน (= ground truth)
cross_platform = df_clean.groupby('profile_id')['platform'].nunique()
multi_platform = cross_platform[cross_platform > 1]
print(f"  Cross-platform IDs : {len(multi_platform)} (มีข้อมูลมากกว่า 1 platform)")

# ตัวอย่าง profile_id
print(f"\n🔍 ตัวอย่าง outputProfileName → profile_id (10 rows):")
print("-" * 60)
sample = df_clean[['outputProfileName', 'profile_id', 'platform']].drop_duplicates('profile_id').head(10)
for _, row in sample.iterrows():
    print(f"  [{row['platform']:10s}] '{row['outputProfileName']}' → '{row['profile_id']}'")

# ตัวอย่าง cross-platform match
if len(multi_platform) > 0:
    print(f"\n🔗 ตัวอย่าง Cross-Platform Match (คนเดียวกันหลาย platform):")
    print("-" * 60)
    sample_ids = multi_platform.head(3).index.tolist()
    for pid in sample_ids:
        rows = df_clean[df_clean['profile_id'] == pid][['platform', 'userName', 'fullName']]
        print(f"  profile_id = '{pid}':")
        for _, r in rows.iterrows():
            print(f"    📱 {r['platform']:10s} | @{r['userName']} | {r['fullName']}")
        print()

print(f"✅ Step 3.8 เสร็จ — เพิ่ม column: profile_id")

### 📋 Stage 3 Summary
สรุปผลรวมของขั้นตอน Data Cleaning ทั้งหมด

In [ ]:
print("=" * 60)
print("📊 STAGE 3 SUMMARY — Data Cleaning & Standardization")
print("=" * 60)

print(f"\n  DataFrame shape: {df_clean.shape}")
print(f"  Columns เพิ่มใหม่: userName_clean, fullName_clean, bio_clean, location_clean, externalUrl_clean, profile_id")

print(f"\n  {'Column':<22s} {'Non-Empty':>10s} {'Pct':>8s}  Bar")
print(f"  {'-'*22} {'-'*10} {'-'*8}  {'-'*20}")

check_cols = [
    ('userName_clean',    'Username (norm)'),
    ('fullName_clean',    'FullName (norm)'),
    ('bio_clean',         'Bio (clean)'),
    ('location_clean',    'Location (clean)'),
    ('externalUrl_clean', 'ExtURL (clean)'),
    ('pictureURL',        'Picture URL'),
    ('profile_id',        'Profile ID'),
]

for col, label in check_cols:
    if col in df_clean.columns:
        non_empty = (df_clean[col].str.len() > 0).sum()
        total = len(df_clean)
        pct = non_empty / total * 100
        bar = chr(9608) * int(pct / 5) + chr(9617) * (20 - int(pct / 5))
        print(f"  {label:<22s} {non_empty:>10,}  {pct:>6.1f}%  {bar}")

print(f"\n  Per-Platform Distribution:")
for platform in df_clean['platform'].unique():
    count = (df_clean['platform'] == platform).sum()
    print(f"    📱 {platform:12s}: {count:>6,} profiles")

print(f"\n{'='*60}")
print(f"✅ Stage 3 COMPLETE — df_clean พร้อมใช้งานใน Stage ถัดไป")
print(f"{'='*60}")

---
## Stage 4: Platform Splitting
แยก DataFrame ออกเป็น **DataFrame แยกตาม platform**  
เพื่อให้สะดวกในการสร้าง cross-platform pairs ในขั้นถัดไป

In [27]:
def split_by_platform(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    แยก DataFrame ตาม platform
    
    Returns:
        Dict เช่น {'twitter': df_tw, 'instagram': df_ig, 'googleplus': df_gp}
    """
    platforms = {}
    
    for platform in df['platform'].unique():
        platform_df = df[df['platform'] == platform].copy()
        platform_df = platform_df.reset_index(drop=True)
        platforms[platform] = platform_df
        print(f"  📱 {platform}: {len(platform_df)} profiles")
    
    return platforms

In [28]:
# === รัน Stage 4 ===
platform_dfs = split_by_platform(df_clean)

print(f"\n{'='*50}")
print(f"📊 Stage 4 Summary")
print(f"{'='*50}")
print(f"Platforms: {list(platform_dfs.keys())}")

for name, pdf in platform_dfs.items():
    non_empty_bio = (pdf['bio_clean'].str.len() > 0).sum()
    non_empty_loc = (pdf['location'].str.len() > 0).sum()
    print(f"\n  [{name}]")
    print(f"    Rows: {len(pdf)}")
    print(f"    Non-empty bio: {non_empty_bio} ({non_empty_bio/len(pdf)*100:.1f}%)")
    print(f"    Non-empty location: {non_empty_loc} ({non_empty_loc/len(pdf)*100:.1f}%)")

  📱 googleplus: 8302 profiles
  📱 twitter: 8535 profiles
  📱 instagram: 7892 profiles

📊 Stage 4 Summary
Platforms: ['googleplus', 'twitter', 'instagram']

  [googleplus]
    Rows: 8302
    Non-empty bio: 5930 (71.4%)
    Non-empty location: 0 (0.0%)

  [twitter]
    Rows: 8535
    Non-empty bio: 8035 (94.1%)
    Non-empty location: 7939 (93.0%)

  [instagram]
    Rows: 7892
    Non-empty bio: 6302 (79.9%)
    Non-empty location: 0 (0.0%)


---
## Stage 5: Ground Truth & Training Data
สร้าง **pairs** สำหรับ cross-platform user matching:
- **Positive pairs** = คนเดียวกันข้ามแพลตฟอร์ม (match โดย `profile_id`)
- **Negative pairs** = คนละคนจากต่าง platform (random sampling)

Platform combinations:
- Twitter ↔ Instagram
- Twitter ↔ Google+
- Instagram ↔ Google+

In [29]:
# --- 5.1 สร้าง Ground Truth Pairs (Positive) ---

def create_ground_truth_pairs(
    df: pd.DataFrame,
    platform1: str = 'twitter',
    platform2: str = 'instagram'
) -> pd.DataFrame:
    """
    สร้าง ground truth pairs สำหรับ cross-platform matching
    ใช้ 'profile_id' (จาก outputProfileName) เป็นตัวจับคู่
    """
    df1 = df[df['platform'] == platform1].copy()
    df2 = df[df['platform'] == platform2].copy()
    
    common_ids = set(df1['profile_id'].unique()) & set(df2['profile_id'].unique())
    print(f"  Found {len(common_ids)} matching users between {platform1} ↔ {platform2}")
    
    pairs = []
    for profile_id in common_ids:
        if not profile_id:
            continue
            
        row1 = df1[df1['profile_id'] == profile_id].iloc[0]
        row2 = df2[df2['profile_id'] == profile_id].iloc[0]
        
        pairs.append({
            'profile_id': profile_id,
            f'{platform1}_userName': row1['userName'],
            f'{platform1}_userName_clean': row1['userName_clean'],
            f'{platform1}_fullName': row1['fullName'],
            f'{platform1}_fullName_clean': row1['fullName_clean'],
            f'{platform1}_bio': row1['bio'],
            f'{platform1}_location': row1.get('location', ''),
            f'{platform2}_userName': row2['userName'],
            f'{platform2}_userName_clean': row2['userName_clean'],
            f'{platform2}_fullName': row2['fullName'],
            f'{platform2}_fullName_clean': row2['fullName_clean'],
            f'{platform2}_bio': row2['bio'],
            f'{platform2}_location': row2.get('location', ''),
            'is_match': 1
        })
    
    return pd.DataFrame(pairs)

print("✅ create_ground_truth_pairs() defined")

✅ create_ground_truth_pairs() defined


In [30]:
# --- 5.2 สร้าง Full Training Dataset (Positive + Negative) ---

def create_full_matching_dataset(
    df: pd.DataFrame,
    platform1: str = 'twitter',
    platform2: str = 'instagram',
    negative_ratio: float = 1.0
) -> pd.DataFrame:
    """
    สร้าง dataset สำหรับ training ที่มีทั้ง positive และ negative pairs
    """
    positive_pairs = create_ground_truth_pairs(df, platform1, platform2)
    n_positive = len(positive_pairs)
    print(f"  ✅ Created {n_positive} positive pairs")
    
    df1 = df[df['platform'] == platform1].copy()
    df2 = df[df['platform'] == platform2].copy()
    
    n_negative = int(n_positive * negative_ratio)
    negative_pairs = []
    
    np.random.seed(42)
    
    attempts = 0
    max_attempts = n_negative * 10
    
    while len(negative_pairs) < n_negative and attempts < max_attempts:
        attempts += 1
        idx1 = np.random.randint(0, len(df1))
        idx2 = np.random.randint(0, len(df2))
        row1 = df1.iloc[idx1]
        row2 = df2.iloc[idx2]
        
        if row1['profile_id'] != row2['profile_id']:
            negative_pairs.append({
                'profile_id': f"neg_{len(negative_pairs)}",
                f'{platform1}_userName': row1['userName'],
                f'{platform1}_userName_clean': row1['userName_clean'],
                f'{platform1}_fullName': row1['fullName'],
                f'{platform1}_fullName_clean': row1['fullName_clean'],
                f'{platform1}_bio': row1['bio'],
                f'{platform1}_location': row1.get('location', ''),
                f'{platform2}_userName': row2['userName'],
                f'{platform2}_userName_clean': row2['userName_clean'],
                f'{platform2}_fullName': row2['fullName'],
                f'{platform2}_fullName_clean': row2['fullName_clean'],
                f'{platform2}_bio': row2['bio'],
                f'{platform2}_location': row2.get('location', ''),
                'is_match': 0
            })
    
    print(f"  ❌ Created {len(negative_pairs)} negative pairs")
    
    negative_df = pd.DataFrame(negative_pairs)
    full_dataset = pd.concat([positive_pairs, negative_df], ignore_index=True)
    full_dataset = full_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return full_dataset

print("✅ create_full_matching_dataset() defined")

✅ create_full_matching_dataset() defined


In [31]:
# === รัน Stage 5 ===

platform_combos = [
    ('twitter', 'instagram'),
    ('twitter', 'googleplus'),
    ('instagram', 'googleplus')
]

# 5A: Ground truth pairs
print("🔗 Creating Ground Truth Pairs...")
print("=" * 50)
pairs = {}
for p1, p2 in platform_combos:
    if p1 in platform_dfs and p2 in platform_dfs:
        pair_key = f"{p1}_{p2}"
        pairs[pair_key] = create_ground_truth_pairs(df_clean, p1, p2)

# 5B: Full training datasets
print(f"\n🏋️ Creating Training Datasets (positive + negative)...")
print("=" * 50)
training_datasets = {}
for p1, p2 in platform_combos:
    if p1 in platform_dfs and p2 in platform_dfs:
        key = f"{p1}_{p2}_training"
        print(f"\n--- {p1} ↔ {p2} ---")
        training_datasets[key] = create_full_matching_dataset(df_clean, p1, p2, negative_ratio=1.0)

print(f"\n{'='*50}")
print(f"📊 Stage 5 Summary")
print(f"{'='*50}")
print(f"Ground truth pairs:")
for k, v in pairs.items():
    print(f"  {k}: {len(v)} pairs")
print(f"\nTraining datasets:")
for k, v in training_datasets.items():
    pos = (v['is_match'] == 1).sum()
    neg = (v['is_match'] == 0).sum()
    print(f"  {k}: {len(v)} total (pos={pos}, neg={neg})")

🔗 Creating Ground Truth Pairs...
  Found 7733 matching users between twitter ↔ instagram
  Found 7733 matching users between twitter ↔ googleplus
  Found 7733 matching users between instagram ↔ googleplus

🏋️ Creating Training Datasets (positive + negative)...

--- twitter ↔ instagram ---
  Found 7733 matching users between twitter ↔ instagram
  ✅ Created 7733 positive pairs
  ❌ Created 7733 negative pairs

--- twitter ↔ googleplus ---
  Found 7733 matching users between twitter ↔ googleplus
  ✅ Created 7733 positive pairs
  ❌ Created 7733 negative pairs

--- instagram ↔ googleplus ---
  Found 7733 matching users between instagram ↔ googleplus
  ✅ Created 7733 positive pairs
  ❌ Created 7733 negative pairs

📊 Stage 5 Summary
Ground truth pairs:
  twitter_instagram: 7733 pairs
  twitter_googleplus: 7733 pairs
  instagram_googleplus: 7733 pairs

Training datasets:
  twitter_instagram_training: 15466 total (pos=7733, neg=7733)
  twitter_googleplus_training: 15466 total (pos=7733, neg=7733

---
## Stage 6: Export & Quality Report
บันทึกไฟล์ output ทั้งหมด + สรุปคุณภาพข้อมูล

In [32]:
# --- 6.1 บันทึกไฟล์ ---

saved_files = []

# All cleaned profiles
path = os.path.join(OUTPUT_DIR, "all_profiles_cleaned.csv")
df_clean.to_csv(path, index=False)
saved_files.append(('all_profiles_cleaned.csv', len(df_clean)))
print(f"💾 Saved: all_profiles_cleaned.csv ({len(df_clean)} rows)")

# Platform-specific DataFrames
for platform, pdf in platform_dfs.items():
    filename = f"df_{platform}.csv"
    pdf.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    saved_files.append((filename, len(pdf)))
    print(f"💾 Saved: {filename} ({len(pdf)} rows)")

# Ground truth pairs
for pair_key, pair_df in pairs.items():
    filename = f"pairs_{pair_key}.csv"
    pair_df.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    saved_files.append((filename, len(pair_df)))
    print(f"💾 Saved: {filename} ({len(pair_df)} rows)")

# Training datasets
for key, tdf in training_datasets.items():
    filename = f"{key}.csv"
    tdf.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    saved_files.append((filename, len(tdf)))
    print(f"💾 Saved: {filename} ({len(tdf)} rows)")

print(f"\n✅ Total files saved: {len(saved_files)}")

💾 Saved: all_profiles_cleaned.csv (24729 rows)
💾 Saved: df_googleplus.csv (8302 rows)
💾 Saved: df_twitter.csv (8535 rows)
💾 Saved: df_instagram.csv (7892 rows)
💾 Saved: pairs_twitter_instagram.csv (7733 rows)
💾 Saved: pairs_twitter_googleplus.csv (7733 rows)
💾 Saved: pairs_instagram_googleplus.csv (7733 rows)
💾 Saved: twitter_instagram_training.csv (15466 rows)
💾 Saved: twitter_googleplus_training.csv (15466 rows)
💾 Saved: instagram_googleplus_training.csv (15466 rows)

✅ Total files saved: 10


In [33]:
# --- 6.2 Data Quality Report ---

print("=" * 60)
print("📋 DATA QUALITY REPORT")
print("=" * 60)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total profiles: {len(df_clean)}")
print(f"Platforms: {list(platform_dfs.keys())}")

# Completeness report
print(f"\n--- Completeness (non-empty values) ---")
check_cols = ['userName_clean', 'fullName_clean', 'bio_clean', 'location', 'externalUrl', 'pictureURL']
for col in check_cols:
    if col in df_clean.columns:
        non_empty = (df_clean[col].str.len() > 0).sum()
        pct = non_empty / len(df_clean) * 100
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f"  {col:20s}: {bar} {pct:5.1f}% ({non_empty}/{len(df_clean)})")

# Platform breakdown
print(f"\n--- Per-Platform Profile Counts ---")
for name, pdf in platform_dfs.items():
    print(f"  {name:12s}: {len(pdf):,} profiles")

# Pairs summary
print(f"\n--- Ground Truth Pairs ---")
for k, v in pairs.items():
    print(f"  {k}: {len(v)} matching users")

# Training summary
print(f"\n--- Training Datasets ---")
for k, v in training_datasets.items():
    pos = (v['is_match'] == 1).sum()
    neg = (v['is_match'] == 0).sum()
    print(f"  {k}: {len(v)} samples (pos={pos}, neg={neg}, ratio={neg/max(pos,1):.2f})")

# Output files
print(f"\n--- Output Files ---")
print(f"  Directory: {os.path.abspath(OUTPUT_DIR)}")
for fname, rows in saved_files:
    print(f"  📄 {fname} ({rows:,} rows)")

print(f"\n{'='*60}")
print(f"✅ PREPROCESSING PIPELINE COMPLETE!")
print(f"{'='*60}")

📋 DATA QUALITY REPORT
Timestamp: 2026-03-09 12:13:18
Total profiles: 24729
Platforms: ['googleplus', 'twitter', 'instagram']

--- Completeness (non-empty values) ---
  userName_clean      : ███████████████████░  99.8% (24668/24729)
  fullName_clean      : ███████████████████░  98.2% (24283/24729)
  bio_clean           : ████████████████░░░░  82.0% (20267/24729)
  location            : ██████░░░░░░░░░░░░░░  32.1% (7939/24729)
  externalUrl         : ████████████████░░░░  84.2% (20832/24729)
  pictureURL          : █████████████░░░░░░░  65.9% (16304/24729)

--- Per-Platform Profile Counts ---
  googleplus  : 8,302 profiles
  twitter     : 8,535 profiles
  instagram   : 7,892 profiles

--- Ground Truth Pairs ---
  twitter_instagram: 7733 matching users
  twitter_googleplus: 7733 matching users
  instagram_googleplus: 7733 matching users

--- Training Datasets ---
  twitter_instagram_training: 15466 samples (pos=7733, neg=7733, ratio=1.00)
  twitter_googleplus_training: 15466 samples (pos=